# Search database inspection

Read-only browser for the run/task trace database (`search.db`). Run all cells from top to bottom; every cell is read-only.

In [1]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Locate the repository regardless of the notebook launch directory."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "search").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(f"Could not find project root from {start}")


# When launched from src/search/script this matches the module's existing
# notebook bootstrap (two directories up); parent discovery also supports Run All
# commands started at the repository root.
PROJECT_ROOT = find_project_root(Path(os.getcwd()).resolve())
TRACE_DB_PATH = PROJECT_ROOT / "search.db"

for path in (TRACE_DB_PATH,):
    if not path.is_file():
        raise FileNotFoundError(f"Database not found: {path}")

# mode=ro makes SQLite reject all write operations from this notebook.
TRACE_CONN = sqlite3.connect(f"file:{TRACE_DB_PATH}?mode=ro", uri=True)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", None)

print(f"Project root: {PROJECT_ROOT}")
print(f"Trace DB (read-only): {TRACE_DB_PATH}")


def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def table_names(conn: sqlite3.Connection) -> list[str]:
    return [
        row[0]
        for row in conn.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type='table' AND name != 'sqlite_sequence' ORDER BY name"
        )
    ]


def show_query(label: str, query: str, conn: sqlite3.Connection, *, head: int | None = None) -> pd.DataFrame:
    df = pd.read_sql_query(query, conn)
    print(f"{label}: {df.shape[0]} rows × {df.shape[1]} columns")
    display(df.head(head) if head is not None else df)
    return df

Project root: /Users/kumo/programming/competitor_product_search
Trace DB (read-only): /Users/kumo/programming/competitor_product_search/search.db


## Summary — row counts

In [2]:
row_counts = []
for database, conn in [("search.db", TRACE_CONN)]:
    for table in table_names(conn):
        row_count = conn.execute(f"SELECT COUNT(*) FROM {quote_identifier(table)}").fetchone()[0]
        row_counts.append({"database": database, "table": table, "row_count": row_count})

summary_df = pd.DataFrame(row_counts).sort_values(["database", "table"]).reset_index(drop=True)
display(summary_df)

,database,table,row_count
0,search.db,attempts,2
1,search.db,candidates,40
2,search.db,llm_calls,2
3,search.db,meta,1
4,search.db,node_events,10
5,search.db,runs,2
6,search.db,tasks,2


## search.db — runs

In [22]:
runs_df = show_query("runs", "SELECT * FROM runs", TRACE_CONN)

runs: 5 rows × 23 columns


,run_id,started_at,finished_at,status,mode,input_file,input_sku_col,output_file,country,website,provider_chain,llm_model,concurrency,serper_max_calls,total_tasks,matched_count,no_match_count,error_count,provider_calls,job_config,pipeline_config,git_commit,error_message
0,3c3103868ccf44a9b0a168cee28777d9,2026-08-17T20:15:43.587329+00:00,2026-08-17T20:15:53.392373+00:00,completed,single,None,None,None,uk,tesco,"duckduckgo,serper",deepseek-v4-flash,1,None,1,1,0,0,"{""duckduckgo"": 2, ""serper"": 0}","{""product_name"": ""Magic Rock Saucery 4 X 330ML"", ""website"": ""tesco"", ""brand"": null, ""country"": ""uk""}","{""search"": {""provider"": [""duckduckgo"", ""serper""], ""k"": 10, ""query_mode"": {""duckduckgo"": ""both"", ""serper"": ""keyword""}, ""strip_parens"": true}, ""domain_map"": {...",7112787,NaN
1,2df4c9c51b9144bbbf4025d94c34bb16,2026-08-17T20:36:52.423010+00:00,2026-08-17T20:36:54.456082+00:00,failed,single,None,None,None,uk,tesco,"duckduckgo,serper",deepseek-v4-flash,1,None,1,0,0,1,"{""duckduckgo"": 2, ""serper"": 0}","{""product_name"": ""Magic Rock Saucery 4 X 330ML"", ""website"": ""tesco"", ""brand"": null, ""country"": ""uk""}","{""search"": {""provider"": [""duckduckgo"", ""serper""], ""k"": 10, ""query_mode"": {""duckduckgo"": ""both"", ""serper"": ""keyword""}, ""strip_parens"": true}, ""domain_map"": {...",7112787,unable to open database file
2,82fc24a67cc143d3bb47495d66b00fdc,2026-08-17T20:37:13.953037+00:00,2026-08-17T20:37:22.074901+00:00,completed,single,None,None,None,uk,tesco,"duckduckgo,serper",deepseek-v4-flash,1,None,1,1,0,0,"{""duckduckgo"": 2, ""serper"": 0}","{""product_name"": ""Magic Rock Saucery 4 X 330ML"", ""website"": ""tesco"", ""brand"": null, ""country"": ""uk""}","{""search"": {""provider"": [""duckduckgo"", ""serper""], ""k"": 10, ""query_mode"": {""duckduckgo"": ""both"", ""serper"": ""keyword""}, ""strip_parens"": true}, ""domain_map"": {...",7112787,NaN
3,eaa236e589f444898261f7bcc47f94d0,2026-08-17T22:10:25.953235+00:00,2026-08-17T22:10:32.717423+00:00,completed,single,None,None,None,uk,tesco,"duckduckgo,serper",deepseek-v4-flash,1,None,1,1,0,0,"{""duckduckgo"": 2, ""serper"": 0}","{""product_name"": ""Magic Rock Saucery 4 X 330ML"", ""website"": ""tesco"", ""brand"": null, ""country"": ""uk""}","{""search"": {""provider"": [""duckduckgo"", ""serper""], ""k"": 10, ""query_mode"": {""duckduckgo"": ""both"", ""serper"": ""keyword""}, ""strip_parens"": true}, ""domain_map"": {...",3ecf9c2,NaN
4,3c8bc874324c4f739186503b83be05a5,2026-08-17T22:11:18.405014+00:00,2026-08-17T22:11:24.919738+00:00,completed,single,None,None,None,uk,tesco,"duckduckgo,serper",deepseek-v4-flash,1,None,1,1,0,0,"{""duckduckgo"": 2, ""serper"": 0}","{""product_name"": ""URBAN FRUIT GENTLY BAKED CHERRIES 75G"", ""website"": ""tesco"", ""brand"": null, ""country"": ""uk""}","{""search"": {""provider"": [""duckduckgo"", ""serper""], ""k"": 10, ""query_mode"": {""duckduckgo"": ""both"", ""serper"": ""keyword""}, ""strip_parens"": true}, ""domain_map"": {...",3ecf9c2,NaN


## search.db — tasks

In [8]:
product_name = "Bonkers Zoomers BBQ Beef Flavour 85g"

In [10]:
query_base = """
select *
from tasks
limit 10
"""

query = """
select final_provider, count(run_id) as run_count
from tasks 
group by final_provider
"""

tasks_df = show_query("tasks", query_base, TRACE_CONN)

tasks: 2 rows × 24 columns


,task_id,run_id,row_index,product_name,product_key,brand_input,website,country,status,verdict,failure_kind,matched_url,matched_title,reason,layer_trace,candidates_considered,final_provider,attempt_count,error_type,error_message,traceback,started_at,finished_at,duration_ms
0,1,3c3103868ccf44a9b0a168cee28777d9,0,Magic Rock Saucery 4 X 330ML,d0234a9c48417bc8e73a4d6f3ca1884e,None,tesco,uk,ok,match,matched,https://www.tesco.com/shop/en-GB/products/305830207,Magic Rock Saucery Session Ipa 4 X 330Ml - Tesco Groceries,Candidate 1 matches the query's Magic Rock Saucery Session IPA 4 x 330ml pack size. (via duckduckgo),"{""domain"": ""pass"", ""brand"": ""pass"", ""numeric"": ""pass"", ""distinguishing"": ""pass""}",20,duckduckgo,1,None,None,None,2026-08-17T20:15:43.589322+00:00,2026-08-17T20:15:53.388691+00:00,9799
1,2,ba2dee3e8aa243ed97792c83c4e10d7e,0,Volvic Mineral Water 12x1L,84b6e42d05bffffdc346d92531d89134,None,amazon.co.uk,uk,ok,match,matched,https://www.amazon.co.uk/Volvic-Natural-Mineral-Water-12/dp/B07G9GVMD3,Volvic Natural Mineral Water 12 x 1L : Amazon.co.uk: Grocery,Candidate 1 is the same 12x1L Volvic Natural Mineral Water product. (via duckduckgo),"{""domain"": ""pass"", ""brand"": ""pass"", ""numeric"": ""pass"", ""distinguishing"": ""pass""}",20,duckduckgo,1,None,None,None,2026-08-21T19:25:05.653981+00:00,2026-08-21T19:25:23.858547+00:00,18204


In [5]:
tasks_df = show_query("tasks", f"SELECT * FROM tasks WHERE product_name like '%{product_name}%'", TRACE_CONN)

tasks: 0 rows × 24 columns


,task_id,run_id,row_index,product_name,product_key,brand_input,website,country,status,verdict,failure_kind,matched_url,matched_title,reason,layer_trace,candidates_considered,final_provider,attempt_count,error_type,error_message,traceback,started_at,finished_at,duration_ms


## search.db — attempts

In [ ]:
attempts_df = show_query("attempts", "SELECT * FROM attempts", TRACE_CONN)

## search.db — node_events

In [7]:
node_events_df = show_query("node_events", "SELECT * FROM node_events limit 10", TRACE_CONN)

node_events: 10 rows × 15 columns


,event_id,attempt_id,task_id,run_id,seq,node,status,error_kind,error_message,traceback,detail,candidates_in,candidates_out,started_at,duration_ms
0,1,1,1,3c3103868ccf44a9b0a168cee28777d9,1,search,ok,None,None,None,{},0,20,2026-08-17T20:15:43.606146+00:00,2818
1,2,1,1,3c3103868ccf44a9b0a168cee28777d9,2,domain_filter,ok,None,None,None,"{""domain_rejects"": {""host"": 10, ""not_product_page"": 5}}",20,5,2026-08-17T20:15:46.425881+00:00,0
2,3,1,1,3c3103868ccf44a9b0a168cee28777d9,3,base_match,ok,None,None,None,{},5,3,2026-08-17T20:15:46.426979+00:00,2281
3,4,1,1,3c3103868ccf44a9b0a168cee28777d9,4,distinguishing,ok,None,None,None,{},3,3,2026-08-17T20:15:48.709700+00:00,4676
4,5,1,1,3c3103868ccf44a9b0a168cee28777d9,5,aggregate,ok,None,None,None,{},3,3,2026-08-17T20:15:53.388016+00:00,0
5,6,2,2,ba2dee3e8aa243ed97792c83c4e10d7e,1,search,ok,None,None,None,{},0,20,2026-08-21T19:25:05.687606+00:00,3422
6,7,2,2,ba2dee3e8aa243ed97792c83c4e10d7e,2,domain_filter,ok,None,None,None,"{""domain_rejects"": {""host"": 9, ""not_product_page"": 2}}",20,9,2026-08-21T19:25:09.112231+00:00,0
7,8,2,2,ba2dee3e8aa243ed97792c83c4e10d7e,3,base_match,ok,None,None,None,{},9,5,2026-08-21T19:25:09.114918+00:00,41
8,9,2,2,ba2dee3e8aa243ed97792c83c4e10d7e,4,distinguishing,ok,None,None,None,{},5,5,2026-08-21T19:25:09.157813+00:00,14697
9,10,2,2,ba2dee3e8aa243ed97792c83c4e10d7e,5,aggregate,ok,None,None,None,{},5,5,2026-08-21T19:25:23.857265+00:00,0


## search.db — candidates

In [ ]:
candidates_df = show_query("candidates", "SELECT * FROM candidates", TRACE_CONN)

## search.db — llm_calls

In [ ]:
llm_calls_df = show_query("llm_calls", "SELECT * FROM llm_calls", TRACE_CONN)

## search.db — meta

In [ ]:
meta_df = show_query("meta", "SELECT * FROM meta", TRACE_CONN)

## search.db — views

### v_errors

In [ ]:
v_errors_df = show_query("v_errors", "SELECT * FROM v_errors", TRACE_CONN)

### v_task_result

In [ ]:
v_task_result_df = show_query("v_task_result", "SELECT * FROM v_task_result", TRACE_CONN)

### v_funnel

In [ ]:
v_funnel_df = show_query("v_funnel", "SELECT * FROM v_funnel", TRACE_CONN)

### v_run_summary

In [ ]:
v_run_summary_df = show_query("v_run_summary", "SELECT * FROM v_run_summary", TRACE_CONN)

## Close connections

The cells above use read-only connections. Run the final cell when you are finished with the notebook.

In [ ]:
TRACE_CONN.close()
print("Read-only database connections closed.")